# REMIX-FND backend on Google Colab

Colab uses **Linux**, so PyTorch / FAISS are usually more stable than on some Mac setups.

**Limits:** Session can disconnect after idle time; not for 24/7 production.

**Two ways to reach the API:**
1. **Colab proxy link** (works while this notebook is open; good for testing in the browser).
2. **ngrok** (optional; gives a shareable URL — requires a free [ngrok](https://ngrok.com) authtoken once).

## 1) Get the project into Colab

**Option A — GitHub:** set your repo URL and run the next cell.

**Option B — Upload:** zip `REMIX_FND_v2` on your computer, then in Colab use *Files → Upload* and unzip:
`!unzip -q REMIX_FND_v2.zip -d /content/`

In [ ]:
# Option A: clone (replace with your fork URL if needed)
%cd /content
!rm -rf REMIX_FND_v2 2>/dev/null
# !git clone https://github.com/YOUR_USER/REMIX_FND_v2.git
# If you use upload instead, ensure code lives at /content/REMIX_FND_v2
%cd /content/REMIX_FND_v2

## 2) Install dependencies (CPU)

First run takes a few minutes.

In [ ]:
%pip install -q fastapi uvicorn[standard] python-multipart pydantic
%pip install -q torch transformers sentence-transformers faiss-cpu pillow pandas numpy tqdm scikit-learn
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

## 3) Upload `best_model.pt` (if missing)

The API expects `models/text_classifier/best_model.pt`. Upload via Colab’s file panel into `REMIX_FND_v2/models/text_classifier/` or train on Colab using `training/scripts/train_text_model.py`.

If the file is missing, the server still starts but uses an **untrained** classifier.

## 4) Start the FastAPI server (background)

In [ ]:
import os, sys, threading, subprocess, time

ROOT = "/content/REMIX_FND_v2"
BACK = f"{ROOT}/backend"
os.chdir(ROOT)

for k, v in [("OMP_NUM_THREADS", "1"), ("MKL_NUM_THREADS", "1"), ("KMP_DUPLICATE_LIB_OK", "TRUE")]:
    os.environ.setdefault(k, v)

def serve():
    subprocess.run(
        [sys.executable, "-m", "uvicorn", "run:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],
        cwd=BACK,
    )

t = threading.Thread(target=serve, daemon=True)
t.start()
time.sleep(25)  # wait for model load
print("If no errors above, server should be on port 8000.")

## 5) Open the API from Colab (built-in proxy)

Run the cell below and click the printed link, or open `/docs` for Swagger.

In [ ]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8000)")
print("API base:", url)
print("Swagger:", url + "docs")
print("Health:", url + "health")

## 6) Optional — public URL with ngrok

1. Sign up at https://ngrok.com and copy your authtoken.
2. Put it in `NGROK_TOKEN` below.
3. Run the cell; use the `https://....ngrok-free.app` URL from your phone or another PC.

In [ ]:
NGROK_TOKEN = ""  # paste: ngrok config add-authtoken ...

if NGROK_TOKEN.strip():
    %pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    pub = ngrok.connect(8000, "http")
    print("Public URL:", pub.public_url)
else:
    print("Set NGROK_TOKEN to enable this cell.")

---

### Other hosts (more stable than Colab for a real demo)

| Service | Notes |
|--------|--------|
| **Hugging Face Spaces** | Docker / Gradio; free tier; good for demos. |
| **Railway / Render / Fly.io** | Connect GitHub repo; set start command: `cd backend && uvicorn run:app --host 0.0.0.0 --port $PORT` and env `PORT`. |
| **Google Cloud Run** | Container from `backend/Dockerfile`; pay per use. |

Your repo already has `backend/Dockerfile` and `docker-compose.yml` for container deploys.